In [1]:
import pandas as pd
import polars as pl
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"
meta_output = '../../../1_allele_collection/3_outputs'

In [3]:
orf_distances = pl.read_csv(f'{outputs_dir}/orf_distances.csv', null_values=".", infer_schema_length=10000)
df = pl.read_csv(f'{outputs_dir}/edges_df.csv', null_values=".", infer_schema_length=10000)


In [4]:
# dedup
df = df.with_columns(
    pl.when(pl.col("aa_change").is_null())
    .then(pl.lit("wt"))
    .otherwise(pl.col("aa_change"))
    .alias("aa_change")
)
df = df.with_columns(
    (pl.col("symbol") + "_" + 
     pl.col("ad_symbol") + "_" + 
     pl.col("aa_change")).alias("ppi_group_clean")
)

df = df.sort("db_mut_id", descending=True).unique(subset=["ppi_group_clean"], keep="first")

df

scoring_pla,scoring_pos,large_plate_name,retest_batch,condition,condition_name,assay_id,media_control,ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,db_source_plate,db_source_well,ad_source_plate,ad_source_well,per_well_integrated_intensity_gray,manual_score_growth,total_colony_area,Children_reExpandedColonies_Count,unique_id,manual_score_growth_updated,contamination,outlier_score,outlier_size_score,outlier_flag,size_score_raw,size_score,ad_symbol,symbol,aa_change,gene_variant,spdi,nt_change,chr_num,nuc_loc,ref_allele,…,g2p_Secondary structure (DSSP 9-state)*,g2p_Secondary structure (PDBe/SIFTS),g2p_Sequence conflict (UniProt),g2p_Sequence uncertainty (UniProt),g2p_Signal (UniProt),g2p_Site (UniProt),g2p_Substrate genes,g2p_Topological domain (UniProt),g2p_Transit peptide (UniProt),g2p_Transmembrane (UniProt),g2p_Ubiquitination,g2p_Zinc finger (UniProt),plddt_f32,rsa_f32,pLDDT_Category,RSA_Category,OMIM_IDs,inheritance_pattern,inheritance_source,disease_modules,disease_module_count,OMIM_disease_names,splicing_variant,db_group,db_condition_group,ppi_group,edge_group,seq_ad,seq_db,db_seq_final_right,manual_correction_applied,consensus_score,aa_score,consensus_manual_score_growth,wt_consensus_score,log2fc,ppi_group_clean
str,str,str,str,str,str,i64,bool,i64,str,i64,i64,str,str,str,str,f64,f64,i64,i64,i64,f64,bool,bool,bool,bool,f64,f64,str,str,str,str,str,str,f64,f64,str,…,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,f64,str,bool,str,str,str,str,i64,i64,str,bool,f64,f64,f64,f64,f64,str
"""VUSAPWT1B1r03_020""","""C03""","""VUSAPWT1B1r03_019_020_021_022_…","""VUSAPWT1B1""","""r09""","""LWH1""",1,false,281,null,281,19,"""VUSMutY8Dh010""","""B01""","""CEGSint_Y8A_1""","""H11""",228.282357,1.0,519,1,281953,1.0,false,false,false,false,118478.543194,110.315217,"""MVK""","""MVK""","""Ile268Val""","""MVK_Ile268Val""","""NC_000012.12:109591273:A:G""","""802A>G""",12.0,1.09591274e8,"""A""",…,"""H (α-helix)""","""Helix""",null,null,null,null,null,null,null,null,null,null,98.47,0.591716,""">90""","""<10""","""260920;610377;175900""","""multiple""","""OMIM""","""Rare genetic systemic or rheum…",6.0,"""HYPER-IgD SYNDROME; MEVALONIC …",false,"""281.0_19.0_VUSAPWT1B1""","""281.0_19.0_LWH1_VUSAPWT1B1""","""281.0_281.0_19.0_VUSAPWT1B1""","""281.0_281.0_VUSAPWT1B1""",1,1,"""Y""",false,138.279887,13.690608,2.0,462.61059,-1.734927,"""MVK_MVK_Ile268Val"""
"""VUSAPWT1B2r05_008""","""A23""","""VUSAPWT1B2r05_007_008_009_010_…","""VUSAPWT1B2""","""r11""","""LWH25""",1,false,100011278,null,4870,0,"""DisWTY8Dh03""","""H04""","""CEGSint_Y8A_5""","""A05""",441.754256,3.0,978,1,122901,2.0,false,false,false,false,432035.662494,402.267842,"""KRT16""","""KRT6A""","""wt""",null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""4870.0_0.0_VUSAPWT1B2""","""4870.0_0.0_LWH25_VUSAPWT1B2""","""100011278.0_4870.0_0.0_VUSAPWT…","""100011278.0_4870.0_VUSAPWT1B2""",1,1,null,false,399.328767,7.287427,3.0,399.328767,0.0,"""KRT6A_KRT16_wt"""
"""VUSAPWT2B1r05_010""","""M19""","""VUSAPWT2B1r05_007_008_009_010_…","""VUSAPWT2B1""","""r17""","""LWH25""",2,false,10064,null,52920,2665,"""VUSMutY8Dh006""","""C11""","""CEGSint_Y8A2_1""","""F12""",302.726806,2.0,611,1,188777,2.0,false,false,false,false,184966.078238,172.221674,"""KCNIP4""","""LITAF""","""Val144Met""","""LITAF_Val144Met""","""NC_000016.10:11549692:C:T""","""430G>A""",16.0,1.1549693e7,"""C""",…,"""E (parallel sheets)""",null,null,null,null,null,null,null,null,null,null,null,92.64,0.0,""">90""","""<10""","""601098""","""autosomal dominant""","""OMIM""","""Rare genetic neurological diso…",1.0,"""CHARCOT-MARIE-TOOTH DISEASE, D…",false,"""52920.0_2665.0_VUSAPWT2B1""","""52920.0_2665.0_LWH25_VUSAPWT2B…","""10064.0_52920.0_2665.0_VUSAPWT…","""10064.0_52920.0_VUSAPWT2B1""",1,1,"""Y""",false,143.495663,1.146285,2.0,263.386038,-0.87162,"""LITAF_KCNIP4_Val144Met"""
"""VUSAPWT1B2r05_002""","""K17""","""VUSAPWT1B2

In [5]:
# keep only relevant columns for this notebook

# keep only relevant columns for this notebook
df[[
    'db_mut_id', 
    'symbol', 
    'ad_symbol', 
    'aa_change',
    'log2fc',
    'clinvar_clnsig_clean_pp_strict'
]].write_csv('./tmp/edges_df_luke.csv')

In [6]:
# Tables for Leo

# df.select(['symbol', 'db_orf_id', 'aa_change', 'db_mut_id', 'ad_symbol', 'ad_orf_id', 'clinvar_clnsig_clean', 'consensus_score', 'wt_consensus_score', 'log2fc']).write_csv(f"{outputs_dir}/ppi_level_scores_minimal_Leo.csv")

# orf_distances[['symbol', 'db_orf_id', 'aa_change', 'db_mut_id', 'gene_variant' ,'clinvar_clnsig_clean', 'norm_dist_rms']].write_csv(f"{outputs_dir}/orf_distances_minimal_Leo.csv")

In [7]:
orf_distances.filter(
    pl.col("clinvar_clnsig_clean_pp_strict").is_in(["Pathogenic", "Benign", "Likely pathogenic", "Likely benign"])
).with_columns(
    pl.when(
        pl.col("clinvar_clnsig_clean_pp_strict").is_in(["Pathogenic", "Likely pathogenic"])
    )
    .then(pl.lit("Positive"))
    .otherwise(pl.lit("Negative"))
    .alias("referenceSet"),
    pl.col("norm_dist_rms").alias("score"),
    pl.col("gene_variant").alias("hgvsp")
).select(
    pl.col(["hgvsp", "score", "referenceSet", "norm_dist_rms", "gene_variant", "clinvar_clnsig_clean_pp_strict"])
).write_csv("../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv")

orf_distances.with_columns(
    pl.col("norm_dist_rms").alias("score"),
    pl.col("gene_variant").alias("hgvs_pro")
).select(pl.col(["hgvs_pro", "score"])).write_csv("../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv")

In [8]:
%%bash
source /opt/homebrew/Caskroom/miniconda/base/etc/profile.d/conda.sh
conda activate py310

Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_bcv" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_bcv.log" --bandwidth "bcv" --printTransitions  --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_nrd0" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_nrd0.log" --bandwidth "nrd0" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_default" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_default.log" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_nrd" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_nrd.log" --bandwidth "nrd" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_sj" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_sj.log" --printTransitions --bandwidth "SJ" --kernel "gaussian" --outlierSuppression 0.1

Loading required package: parallel
Loading required package: maveLLR

Attaching package: ‘maveLLR’

The following objects are masked _by_ ‘.GlobalEnv’:

    buildLLR.gauss, buildLLR.kernel, drawDensityLLR

Warning message:
NAs introduced by coercion 


2026-01-14_18:14:52 INFO: Reading input 
2026-01-14_18:14:52 INFO: Calculating densties 


Warning message:
In stats::bw.bcv(data) : minimum occurred at one end of the range


2026-01-14_18:14:53 INFO: Drawing LLR plot 
2026-01-14_18:14:53 INFO: Drawing PRC curve 
  |======================================================================| 100%2026-01-14_18:14:53 INFO: Category transitions:  
0.542 : none -> patho.support
0.643 : patho.support -> patho.moderate
0.715 : patho.moderate -> patho.support
2026-01-14_18:14:54 INFO: Bootstrapping confidence intervals at  10000  iterations 
2026-01-14_18:14:54 INFO: Writing output to file  ../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_bcv.csv 
2026-01-14_18:14:54 INFO: Done! 


In [10]:
LLR_threshold = 0.542

# add ppi_hit column

orf_distances = orf_distances.with_columns(
    pl.when(pl.col("norm_dist_rms") >= LLR_threshold)
    .then(True)
    .otherwise(False)
    .alias("ppi_hit")
)

cols_to_keep = ['db_orf_id',
 'db_mut_id',
 'num_edges',
 'clinvar_clnsig_clean',
 'norm_dist_rms',
 'symbol',
 'gene_variant',
 'aa_change',
 'clinvar_clnsig_clean_pp_strict',
 'ppi_hit']

orf_distances = orf_distances.select(cols_to_keep)

orf_distances.write_csv(f'{outputs_dir}/orf_distances_final.csv')